# 어휘 분해 — 통제군이 학습에서 본 적 없는 단어에서 얼마나 나쁜가

`eval_adapters_big7_colab.ipynb`가 드라이브에 저장한 예측만 읽는다. **재평가 없음, GPU 불필요, 몇 초.**

- 런타임 유형은 **CPU**면 된다(T4를 낭비하지 마라).
- 준비물: 드라이브에 `whisper-608-control-big7-dev.json`과
  `whisper-608-control-big7-preds-{11차,7차재현,10차}.json`(평가 노트북이 만든 것).
- 아래 셀 하나를 실행하면 된다. 드라이브 마운트는 셀이 알아서 한다.
- 표 아래의 **경고** 줄이 뜨면 그 행은 믿지 마라(앞 노트북이 낸 단어 CER과 안 맞는 것이다).


In [ ]:
# ══ 어휘 분해 — 통제군이 학습에서 본 적 있는 단어와 없는 단어를 갈라 본다 ══
# 저장해 둔 예측(whisper-608-control-big7-preds-*.json)만 읽는다. 재평가가 없다.
#
# 왜 보나: 608 단어는 닫힌 어휘(dev 단어의 99.8%가 big7 학습 어휘 안)라서,
# 11차의 단어 이득에는 "이 단어들을 33번씩 봤다"가 섞여 있다. 통제군(big6)은
# 학습 어휘가 203종이라 dev 단어의 9.4%가 **통제군에게는 처음 보는 단어**다.
# 그 부분에서 통제군이 얼마나 나쁜지가 "어휘 밖 단어에서 파인튜닝 모델이 어떻게
# 되는가"의 (약한) 단서다. 앱 표적 단어의 85.6%가 608 어휘 밖이다(대장 참고).
import json, os

BIG6_VOCAB = set("""가 가위 감 강 개 거울 걸 겁 겉 곰 공 과일 구 국 군 권투 귀 귤 그림 극장 글 금 기차 김 까 껌 꽃 꿀 꿈 나 나무 날개 남 냄새 네 노래 녹 논 논밭 눈 느낌 능 다 달 달걀 당근 덕 도 돈 돌 동생 등대 따 딸기 땅콩 떡 똥 뜰 라 마 마음 만 말 모자 목표 몸 무 문 문제 물 밀 바 바다 발 밤 밥 방 방석 배 뱀 벌 별 병 병원 북 불 붓 빗 빠 빵 뼈 뿔 사 사과 사람 산 삼 상 새 성 소 손 송곳 시 시간 신발 실 쌀 아 아들 안개 안경 알 앞 야 약 약국 양 어 에 여 연 연필 영 예 오 옷 와 왕 왜 요 욕심 용 우 우산 웨 위 유 육 육군 으 은 이 일 입 잎 자 자연 잔 장갑 잼 저녁 전화 절 점 접시 조 종 죄 쥐 집 짜 차 참새 창 창문 초 총 추 춤 칠 침 칼 컵 코 콩 키 타 탈 탑 터 턱 털 토끼 톱 파 팔 팥죽 편지 폐 풀 풍선 피 하 학교 해 허리 혀 형 호박 활 회사 흙""".split())   # big6 학습 단어 203종
PRED = "/content/drive/MyDrive/whisper-608-control-big7-preds-{}.json"
# 이 셀은 혼자서도 돈다 — 런타임이 새로 떴으면 드라이브부터 붙인다(GPU 불필요, CPU 런타임이면 된다).
if not os.path.isdir(os.path.dirname(PRED)):
    from google.colab import drive
    drive.mount("/content/drive")
RES = json.load(open("/content/drive/MyDrive/whisper-608-control-big7-dev.json", encoding="utf-8"))


# CER은 앞 셀이 쓴 것과 **같은 라이브러리**(evaluate)로 잰다. 직접 짠 편집거리로
# 대신했더니 통제군 행이 0.0125·0.0060 어긋났다: evaluate는 앞뒤 공백을 지우는데
# Whisper 출력에는 앞 공백(" 거울")이 자주 붙어서, 직접 계산은 그걸 삽입 오류로 셌다.
try:
    import evaluate
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "evaluate", "jiwer"], check=True)
    import evaluate
_cer_metric = evaluate.load("cer")


def _cer(rows):
    if not rows:
        return float("nan")
    return _cer_metric.compute(predictions=[r["hyp"] for r in rows], references=[r["ref"] for r in rows])


print("어댑터".ljust(10) + "단어 전체".rjust(12) + "통제어휘 안".rjust(14) + "통제어휘 밖".rjust(14) + "   (n 전체/안/밖)")
show = {}
for name in ("11차", "7차재현", "10차"):
    path = PRED.format(name)
    if not os.path.exists(path):
        print(name.ljust(10) + "예측 파일 없음 — 건너뜀")
        continue
    rows = [r for r in json.load(open(path, encoding="utf-8")) if r["task"] == "wordlist"]
    inv = [r for r in rows if r["ref"].strip() in BIG6_VOCAB]
    oov = [r for r in rows if r["ref"].strip() not in BIG6_VOCAB]
    # 계산 방식 검증: 앞 셀이 낸 단어 CER과 같아야 이 표를 믿을 수 있다
    diff = abs(_cer(rows) - RES[name]["단어"]["cer"])
    flag = "" if diff < 5e-4 else f"   경고: 앞 셀 값과 {diff:.4f} 어긋남 — 이 행을 믿지 마라"
    print(name.ljust(10) + f"{_cer(rows):12.4f}{_cer(inv):14.4f}{_cer(oov):14.4f}   ({len(rows)} / {len(inv)} / {len(oov)}){flag}")
    show[name] = oov

# 통제군이 처음 보는 단어를 어떻게 틀리는지 — 학습 어휘 안의 비슷한 단어로 끌려가는지 본다
for name in ("7차재현", "10차"):
    if name in show:
        wrong = [r for r in show[name] if r["hyp"].strip() != r["ref"].strip()]
        print()
        print(f"[{name}] 어휘 밖 단어 {len(show[name])}개 중 틀린 것 {len(wrong)}개, 예:")
        for r in wrong[:10]:
            tag = "(학습 어휘 안의 단어로 인식)" if r["hyp"].strip() in BIG6_VOCAB else ""
            print(f"   정답 {str(r['ref']):6} -> 인식 {str(r['hyp']):8} {tag}")
